# M4 exploration: does wind change airborne traffic near Turkish airports?

**Reading only.** Every number comes from the M3 clean tables through the saved queries in
`sql/queries/02-04` and the helpers in `src/explore.py`. The database is opened read-only and
fingerprinted before and after (last cell): if any table changed, the notebook fails.

**Question (from the README):** do stronger gusts go together with more or faster traffic
near an airport?

**What the data can and cannot say.** There are only a handful of ADS-B snapshots, taken on
irregular days. The honest reading of any result below is *exploratory*: it shows what the
data looks like and which question a larger sample could answer.

**Decisions fixed before looking at any traffic outcome**

| Decision | Value | Why |
|---|---|---|
| Airport area | 0.25 deg, about 28 km, nearest airport wins | M2 rule, now without double counting |
| Gust filter | `ok` + `unusual_high` (sensitivity: `ok` only) | `unusual_high` marks extreme but plausible gusts (the 27 Oct storm, ordinary ESB afternoons); `ok` only cuts the tail and thins ESB and AYT (see 1b), so it is the check |
| Forecast hours | excluded (invariant I-20) | a forecast is not an observation |
| Clock | UTC in the database, `Europe/Istanbul` (UTC+3) for every day or hour shown | people fly by local time |
| "Windy" | gust at or above the airport's own 75th percentile of trusted hours (sensitivity: median) | airports differ: a 17 km/h gust is the strongest hour at Ankara but an ordinary one at Istanbul |
| Outcome | airborne aircraft per snapshot, and mean airborne speed, each minus that airport's mean | removes the airport-size effect |
| Independent unit | local calendar day | snapshots on the same day are not independent |
| Local-day coverage rule | at least 18 of 24 hours with a trusted gust | 75% |

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "explore.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import explore

# --- every parameter of the analysis, in one place ---
RADIUS_DEG = 0.25
LON_SCALE = 0.76                      # cos(40.5 deg): one degree of longitude is shorter than one of latitude
GUST_FLAGS = ("ok", "unusual_high")          # reported filter, stated in every query and caption
GUST_FLAGS_STRICT = ("ok",)                    # sensitivity check
MIN_SHARE = 0.75                      # share of the 24 local hours that must carry a trusted gust
WINDY_Q, WINDY_Q_ALT = 0.75, 0.50     # per-airport quantile that defines "windy"
MIN_UNITS = 8                         # fewer independent days than this: exploratory only
N_BOOT = int(os.environ.get("M4_N_BOOT", 5000))
TAIL_Q = 0.90                        # tail summary for 2.2: 90th percentile of the hourly gust
STRONG_GUST = 30.0                   # km/h, fixed threshold for the share-of-strong-hours summary
BIN_WIDTHS = (1.0, 2.5, 5.0)         # km/h, bin widths tried for the gust histogram (2.3)
BIN_WIDTH = 2.5                      # km/h, the one used in the saved figure
WINDY_DAY_MEAN = 20.0                # km/h, a local day counts as windy when its mean gust reaches this (2.4 example)
HOLIDAYS = {"2026-10-28": "eve of Republic Day (half day)", "2026-10-29": "Republic Day"}   # Turkish public holidays in the data span (2.5)
MIN_AIRBORNE_SPEED = 3               # a mean airborne speed needs at least this many aircraft (2.6)
TIME_BAND_EDGES = (11.0, 15.0, 19.0, 23.0)   # local hours; the snapshots were taken between 11:34 and 22:38 (2.5)
SEED = 20261103
TRAFFIC_PARAMS = {"radius_deg": RADIUS_DEG, "lon_scale": LON_SCALE}
CLOCK = "clock: Europe/Istanbul (UTC+3)"

con = explore.connect_ro()
fp_before = explore.db_fingerprint(con)
print("database opened read-only; fingerprint taken for", len(fp_before), "tables")

database opened read-only; fingerprint taken for 8 tables


## 0. Raw versus cleaned: the key number (M4 2.1)

**Key number:** the mean wind gust, in km/h, at the five airports (the weather side of the
README question). We compute it twice: on the M2 table `weather_hourly` as loaded
(query `05`), and on `weather_hourly_clean` with the flag filter we use, `ok` + `unusual_high`, and
with forecast hours removed (query `06`). Two intermediate steps separate the effect of the
forecast rule (I-20) from the effect of the flags; the strict filter `ok` only is shown as a check. No clock is involved: nothing is grouped by time.

In [2]:
print("-- query 05 (raw, M2 table) --")
print(explore.load_sql("05_gust_mean_raw"))
print("-- query 06 (clean, flag filter bound as :f1..:f3) --")
print(explore.load_sql("06_gust_mean_clean"))

-- query 05 (raw, M2 table) --


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ACER\\flight-delay-weather-analysis\\sql\\queries\\05_gust_mean_raw.sql'

In [ ]:
STEPS = {
    "1 raw, all hours": ("05_gust_mean_raw", {"observed_only": 0}),
    "2 raw, forecast hours removed": ("05_gust_mean_raw", {"observed_only": 1}),
    "3 clean, flag ok + unusual_high (reported)": ("06_gust_mean_clean", explore.flag_params(GUST_FLAGS)),
    "4 clean, flag ok only (strict)": ("06_gust_mean_clean", explore.flag_params(GUST_FLAGS_STRICT)),
}
frames = []
for label, (name, params) in STEPS.items():
    f = explore.query_df(con, name, params)
    f.insert(0, "step", label)
    frames.append(f)
raw_vs_clean = pd.concat(frames, ignore_index=True)

print("Mean gust, km/h (n = hours with a gust value in brackets below)")
display(raw_vs_clean.pivot(index="iata", columns="step", values="mean_gust_kmh")[list(STEPS)])
display(raw_vs_clean.pivot(index="iata", columns="step", values="n_gust")[list(STEPS)])

In [ ]:
a, b, c, d = (raw_vs_clean[raw_vs_clean.iata == "ALL"].set_index("step").loc[list(STEPS)].itertuples())
esb = raw_vs_clean[raw_vs_clean.iata == "ESB"].set_index("step").mean_gust_kmh
REPORTED, STRICT = "3 clean, flag ok + unusual_high (reported)", "4 clean, flag ok only (strict)"
print(f"Mean gust, all airports pooled: raw {a.mean_gust_kmh:.2f} km/h (n = {a.n_gust}) -> clean {c.mean_gust_kmh:.2f} km/h "
      f"(n = {c.n_gust}), a change of {c.mean_gust_kmh - a.mean_gust_kmh:+.2f}.")
print(f"- Removing forecast hours (I-20) moves it by {b.mean_gust_kmh - a.mean_gust_kmh:+.2f} "
      f"and drops {a.n_gust - b.n_gust} hours.")
print(f"- The reported flag filter moves it by {c.mean_gust_kmh - b.mean_gust_kmh:+.2f} and drops {b.n_gust - c.n_gust} more hour(s) "
      f"(stuck_value or invalid); the strict filter 'ok' only would give {d.mean_gust_kmh:.2f} "
      f"({d.mean_gust_kmh - b.mean_gust_kmh:+.2f}, {b.n_gust - d.n_gust} hours dropped, almost all unusual_high).")
print(f"- The strict filter costs the airports unevenly: ESB would fall from {esb.iloc[0]:.2f} to {esb[STRICT]:.2f} km/h "
      f"(reported filter: {esb[REPORTED]:.2f}), because most unusual_high hours are ordinary daytime gusts at ESB (I-11).")
means = raw_vs_clean[raw_vs_clean.iata != "ALL"].pivot(index="iata", columns="step", values="mean_gust_kmh")
same = list(means["1 raw, all hours"].sort_values().index) == list(means[REPORTED].sort_values().index)
print("- Airport ranking by mean gust is", "unchanged" if same else "DIFFERENT", "between raw and clean.")


## 0b. The question and the summary that answers it (M4 2.2)

**Question:** do stronger gusts go together with more or faster airborne traffic near an airport?

The weather half of that question is about *strong* hours, not typical ones, so the summary
of the gust variable is the **90th percentile of the hourly gust** per airport (the level that
the windiest tenth of hours reaches; about 33 hours per airport). The mean is dominated by calm
hours, and the maximum rests on a single hour. The alternative summaries are the **median** (the
typical hour), the mean, the 95th percentile and the **share of hours with a gust of at least
`STRONG_GUST` km/h**. Every value carries its n, the number of hours with a gust that passed the
flag filter.

**Flag filter: `ok` + `unusual_high`, the reported filter of this notebook.** `unusual_high` hours are the top
of each airport's own distribution (I-11), so under `ok` only the upper tail is cut off by
construction and a tail summary would describe the filter, not the wind. Both filters are shown;
the `ok`-only result is the sensitivity check. Clock: not used (no time grouping). Forecast hours
are excluded (I-20).

In [ ]:
def gust_summary(frame):
    g = frame[frame.gust_usable == 1].groupby("iata").wind_gusts_10m
    return pd.DataFrame({
        "n": g.count(),
        "mean": g.mean(),
        "median": g.median(),
        f"p{int(TAIL_Q * 100)}": g.quantile(TAIL_Q),
        "p95": g.quantile(0.95),
        f"share >= {STRONG_GUST:g} km/h": g.apply(lambda s: (s >= STRONG_GUST).mean()),
        f"n >= {STRONG_GUST:g} km/h": g.apply(lambda s: int((s >= STRONG_GUST).sum())),
    })

summ = {}
for label, flags in {"ok + unusual_high (reported)": GUST_FLAGS, "ok only (sensitivity)": GUST_FLAGS_STRICT}.items():
    frame = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(flags))
    summ[label] = gust_summary(frame)
    print(f"Gust summaries in km/h per airport; flag filter: {label}; forecast hours excluded")
    display(summ[label].round(2))

In [ ]:
TAIL = f"p{int(TAIL_Q * 100)}"
SHARE = f"share >= {STRONG_GUST:g} km/h"
for label, table in summ.items():
    ranks = table[["mean", "median", TAIL, "p95", SHARE]].rank(ascending=False, method="min").astype(int)
    print(f"Rank by summary (1 = windiest; ties share a rank); flag filter: {label}")
    display(ranks)

main = summ["ok + unusual_high (reported)"]
print(f"Ranking by {TAIL} (reported):", " > ".join(main[TAIL].sort_values(ascending=False).index))
print("Ranking by median:       ", " > ".join(main["median"].sort_values(ascending=False).index))
for a_ in main.index:
    for b_ in main.index:
        if a_ < b_ and (main.loc[a_, "median"] - main.loc[b_, "median"]) * (main.loc[a_, TAIL] - main.loc[b_, TAIL]) < 0:
            hi_med = a_ if main.loc[a_, "median"] > main.loc[b_, "median"] else b_
            hi_tail = b_ if hi_med == a_ else a_
            print(f"- {hi_med} has the higher median ({main.loc[hi_med, 'median']:.1f} vs {main.loc[hi_tail, 'median']:.1f} km/h) "
                  f"but {hi_tail} the higher {TAIL} ({main.loc[hi_tail, TAIL]:.1f} vs {main.loc[hi_med, TAIL]:.1f} km/h): "
                  f"the median answers 'which airport is windier on a typical hour', the {TAIL} 'which reaches strong gusts more often'.")
ok_only = summ["ok only (sensitivity)"]
same_as_ok = list(ok_only[TAIL].sort_values(ascending=False).index) == list(main[TAIL].sort_values(ascending=False).index)
print(f"- Under the `ok`-only filter the {TAIL} ranking is", "the same." if same_as_ok else
      "different: " + " > ".join(ok_only[TAIL].sort_values(ascending=False).index) + " (the filter removes the tail).")

## 0c. Shape of the key variable (M4 2.3)

**Question:** what shape do hourly gusts have, overall and at each airport, and what does that mean
for the summary chosen in 0b?

Two views that do not share a weakness: a **histogram** (shows modes and the body, but depends on
the bin width) and an **ECDF** (no binning, shows every quantile at once, but modes are hard to
see). Bin widths of 1, 2.5 and 5 km/h are tried first on the pooled data (`BIN_WIDTHS`; trial plot
below, not saved). Flag filter: `ok` + `unusual_high` (as in 0b). Forecast hours excluded.
Clock: not used.

In [ ]:
shape = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(GUST_FLAGS))
shape = shape[shape.gust_usable == 1].copy()
vals = shape.wind_gusts_10m

# trial plot (not saved to figures/): chooses the bin width for the saved figure fig05
fig, axes = plt.subplots(1, len(BIN_WIDTHS), figsize=(12, 3), sharex=True)
for ax, bw in zip(axes, BIN_WIDTHS):
    ax.hist(vals, bins=np.arange(0, vals.max() + bw, bw), color="#4c72b0")
    ax.set_title(f"bin width {bw:g} km/h", fontsize=9)
    ax.set_xlabel("wind gust at 10 m, km/h")
axes[0].set_ylabel("hours per bin")
fig.suptitle(f"Bin-width trial, all airports pooled (n = {len(vals)} hours; flag filter: {', '.join(GUST_FLAGS)}; trial, not saved)",
             fontsize=9)
fig.tight_layout()
plt.show()

# Does the number of modes depend on the bin width? Count local maxima of the 3-bin smoothed histogram.
def n_peaks(x, bw):
    h, _ = np.histogram(x, bins=np.arange(0, x.max() + bw, bw))
    sm = np.convolve(h, [1, 1, 1], "same")
    return int(sum(sm[i] > sm[i - 1] and sm[i] >= sm[i + 1] and sm[i] > 0.08 * sm.max() for i in range(1, len(sm) - 1)))
print(pd.DataFrame({f"{bw:g} km/h": {k: n_peaks(v, bw) for k, v in
                    [("all", vals)] + [(i, p.wind_gusts_10m) for i, p in shape.groupby("iata")]}
                    for bw in BIN_WIDTHS}).rename_axis("peaks found").to_string())

**Bin width.** At 1 km/h the histogram of about 330 hours per airport is noise: it finds 4 to 8
"peaks" per airport, mostly single bins. At 5 km/h the body of the distribution is two or three
bars and nothing else can be seen. We use **2.5 km/h** (about 1.3 knots, wider than the 0.1 km/h
resolution of the data): ADB, ESB and SAW then have one main peak, and the right tail is still
visible. AYT and IST keep weaker extra bumps (a few hours per bin) that we do not interpret.

In [ ]:
airports = sorted(shape.iata.unique())
colors = dict(zip(airports, plt.cm.tab10.colors))
edges = np.arange(0, shape.wind_gusts_10m.max() + BIN_WIDTH, BIN_WIDTH)
xmax = float(edges[-1])

fig = plt.figure(figsize=(12, 7.2))
gs = fig.add_gridspec(2, 5, height_ratios=[1, 1.15])
top = [fig.add_subplot(gs[0, 0])]
top += [fig.add_subplot(gs[0, i], sharex=top[0], sharey=top[0]) for i in range(1, 5)]
for ax, iata in zip(top, airports):
    x = shape.loc[shape.iata == iata, "wind_gusts_10m"]
    ax.hist(x, bins=edges, color=colors[iata])
    ax.axvline(x.median(), color="black", lw=1, ls="--")
    ax.axvline(x.quantile(TAIL_Q), color="black", lw=1, ls=":")
    ax.set_title(f"{iata} (n = {len(x)})", fontsize=10)
    ax.set_xlabel("gust, km/h")
    ax.set_xlim(0, xmax)
top[0].set_ylabel("hours per 2.5 km/h bin")
for ax in top[1:]:
    plt.setp(ax.get_yticklabels(), visible=False)
top[-1].plot([], [], "k--", lw=1, label="median")
top[-1].plot([], [], "k:", lw=1, label=f"{int(TAIL_Q * 100)}th percentile")
top[-1].legend(fontsize=8, loc="upper right")

ax_h = fig.add_subplot(gs[1, 0:2])
ax_h.hist(vals, bins=edges, color="#888888")
ax_h.axvline(vals.median(), color="black", lw=1, ls="--")
ax_h.axvline(vals.quantile(TAIL_Q), color="black", lw=1, ls=":")
ax_h.axvline(STRONG_GUST, color="#c44e52", lw=1.2)
ax_h.set_title(f"All airports pooled (n = {len(vals)})", fontsize=10)
ax_h.set_xlabel("wind gust at 10 m, km/h")
ax_h.set_ylabel(f"hours per {BIN_WIDTH:g} km/h bin")
ax_h.set_xlim(0, xmax)

ax_e = fig.add_subplot(gs[1, 2:5])
for iata in airports:
    x = np.sort(shape.loc[shape.iata == iata, "wind_gusts_10m"].to_numpy())
    ax_e.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=colors[iata], label=iata)
xs = np.sort(vals.to_numpy())
ax_e.step(xs, np.arange(1, len(xs) + 1) / len(xs), where="post", color="#888888", lw=2.2, label="all")
ax_e.axvline(STRONG_GUST, color="#c44e52", lw=1.2)
ax_e.axhline(TAIL_Q, color="black", lw=0.8, ls=":")
ax_e.text(STRONG_GUST + 0.6, 0.04, f"{STRONG_GUST:g} km/h", color="#c44e52", fontsize=8)
ax_e.set_title("ECDF: share of hours with a gust at or below x", fontsize=10)
ax_e.set_xlabel("wind gust at 10 m, km/h")
ax_e.set_ylabel("share of hours")
ax_e.set_xlim(0, xmax)
ax_e.set_ylim(0, 1.01)
ax_e.legend(title="airport", fontsize=8, ncols=2, loc="lower right")

fig.suptitle("Gusts are right-skewed at every airport: calm hours dominate, with a long tail of strong ones",
             fontsize=13, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.945, "Question: what shape do hourly gusts have, and what does that mean for the summary we report?",
         fontsize=10, color="#444444")
explore.save_fig(fig, "fig05_gust_shape",
                 f"n = {len(vals)} hours ({', '.join(f'{a} {int((shape.iata == a).sum())}' for a in airports)}); "
                 f"flag filter: {', '.join(GUST_FLAGS)}; forecast hours excluded; bin width {BIN_WIDTH:g} km/h; "
                 "clock: not used (no time grouping); red line = 30 km/h strong-gust threshold", top=0.91)
plt.show()

In [ ]:
g = shape.groupby("iata").wind_gusts_10m
shape_tab = pd.DataFrame({"n": g.count(), "skewness": g.skew(), "mean": g.mean(), "median": g.median(),
                          "mean - median": g.mean() - g.median(), TAIL: g.quantile(TAIL_Q), "max": g.max()})
display(shape_tab.round(2))
print(f"- Skewness is positive at {int((shape_tab.skewness > 0).sum())} of {len(shape_tab)} airports "
      f"({shape_tab.skewness.min():.1f} to {shape_tab.skewness.max():.1f}); the mean is above the median at "
      f"{int((shape_tab['mean - median'] > 0).sum())} of {len(shape_tab)}.")
top_hours = shape[shape.wind_gusts_10m >= 50].assign(local_date=lambda d: explore.to_local(d.observed_at).dt.strftime("%Y-%m-%d"))
print(f"- {len(top_hours)} hours reach 50 km/h or more, at {', '.join(sorted(top_hours.iata.unique()))}, on local day(s) "
      f"{', '.join(sorted(top_hours.local_date.unique()))}: the far tail is one event, not a regular feature.")

**What we see.** Gusts are **right-skewed with a long upper tail** at every airport; there is no
clipping at a limit. Each airport has one main peak in the calm range, with small secondary
bumps at AYT and IST that are too few hours to read as separate regimes. Istanbul is shifted to
the right of everything else, and its largest hours form a small separate cluster (a single storm
day, printed above). Ankara is the calmest, with the most skewed distribution: a low typical hour
and an occasional strong one.

**What it means for the summary.** Because of the skew, the mean sits above the median at every
airport and is pulled up by a few strong hours, so it would overstate a typical hour. The median
describes the typical hour; the 90th percentile describes the tail and is the level that defines a
"windy" hour in this project. The 90th percentile is stable here (about 33 hours above it per
airport), whereas the maximum rests on one hour and on one storm. The ECDF also shows that the
30 km/h line separates the airports: it lies in the extreme tail at ESB and ADB but well inside
the distribution at IST and SAW.

## 1. Weather supply: how many trusted hours exist?

Query `03` returns every observed hour (forecasts removed) and keeps rows whose gust is not
trusted, with the value set to NULL, so the denominators stay visible. We then count per
**local** day, because a UTC day cuts a Turkish day in two (the UTC table from query `04` is
kept as the data-supply view).

In [ ]:
w = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(GUST_FLAGS))
w["local_time"] = explore.to_local(w.observed_at)
w["local_date"] = w.local_time.dt.strftime("%Y-%m-%d")

cov_utc = explore.query_df(con, "04_coverage_weather", explore.flag_params(GUST_FLAGS))
cov_local = explore.coverage_table(w, ["iata", "local_date"], "wind_gusts_10m",
                                   expected=24, min_share=MIN_SHARE)

print(f"{len(w)} observed weather hours, {int(w.gust_usable.sum())} with a trusted gust "
      f"({w.gust_usable.mean():.1%}); forecast hours already removed")
print(f"UTC days with data: {cov_utc.utc_date.nunique()}; local days with data: {cov_local.local_date.nunique()}")
summary = (cov_local.groupby("iata")
           .agg(days=("local_date", "size"), days_meeting_rule=("meets_rule", "sum"),
                trusted_hours=("n", "sum")))
summary

In [ ]:
strict = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(GUST_FLAGS_STRICT))
sens = pd.DataFrame({
    "hours": w.groupby("iata").size(),
    "gust_reported_filter": w.groupby("iata").gust_usable.sum(),
    "gust_strict_ok_only": strict.groupby("iata").gust_usable.sum(),
})
sens["hours_lost_by_strict_filter"] = sens.gust_reported_filter - sens.gust_strict_ok_only
sens["max_gust_reported"] = w.groupby("iata").wind_gusts_10m.max()
sens["max_gust_strict"] = strict.groupby("iata").wind_gusts_10m.max()
sens

In [ ]:
dates = [d.strftime("%Y-%m-%d") for d in
         pd.date_range(cov_local.local_date.min(), cov_local.local_date.max(), freq="D")]
airports = sorted(cov_local.iata.unique())
grid = (cov_local.pivot(index="iata", columns="local_date", values="share")
        .reindex(index=airports, columns=dates))

fig, ax = plt.subplots(figsize=(11, 4.2))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("#e6e6e6")               # days with no weather row at all
im = ax.imshow(np.ma.masked_invalid(grid.to_numpy(dtype=float)), aspect="auto", vmin=0, vmax=1, cmap=cmap)
ax.set_yticks(range(len(airports)), airports)
ax.set_xticks(range(len(dates)), [d[5:] for d in dates], rotation=90, fontsize=7)
ax.set_xlabel("local date, 2026 (Europe/Istanbul)")
ax.set_ylabel("airport")
thin = cov_local[~cov_local.meets_rule]                      # below the coverage rule: outlined
for r_ in thin.itertuples():
    ax.add_patch(plt.Rectangle((dates.index(r_.local_date) - 0.5, airports.index(r_.iata) - 0.5), 1, 1,
                               fill=False, ec="#c44e52", lw=1.8))
cbar = fig.colorbar(im, ax=ax, label="share of the 24 local hours")
cbar.ax.axhline(MIN_SHARE, color="#c44e52", lw=2)
cbar.set_ticks([0, 0.25, 0.5, MIN_SHARE, 1])
cbar.set_ticklabels(["0", "0.25", "0.5", f"{MIN_SHARE:g} = {int(MIN_SHARE * 24)} h", "1"])
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fill=False, ec="#c44e52", lw=1.8, label=f"below the {int(MIN_SHARE * 24)}-of-24-hours rule"),
                   plt.Rectangle((0, 0), 1, 1, fc="#e6e6e6", label="no data that day")],
          loc="lower right", bbox_to_anchor=(1.0, 1.01), ncols=2, fontsize=8, frameon=False)

days_failing = thin.groupby("local_date").iata.nunique()
shared = [d_ for d_, k_ in days_failing.items() if k_ == len(airports)]
def short(d_):
    t_ = pd.Timestamp(d_)
    return f"{t_.day} {t_:%b}"
if len(thin) and set(thin.local_date) == set(shared):
    headline = f"All {len(airports)} airports share the same {len(shared)} thin days ({' and '.join(short(d_) for d_ in shared)}); every other day passes the rule"
elif len(thin):
    headline = f"{len(thin)} of {len(cov_local)} airport-days fall below the {int(MIN_SHARE * 24)}-of-24-hours rule"
else:
    headline = "Every airport-day passes the coverage rule"
fig.suptitle(headline, fontsize=12, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.905, "Question: how complete is the weather record, by airport and local day, against the 24 hours that could exist?",
         fontsize=9.5, color="#444444")
explore.save_fig(fig, "fig01_weather_coverage",
                 f"n = {int(w.gust_usable.sum())} trusted hours of {len(w)} observed hours (forecast hours excluded); "
                 f"flag filter: {', '.join(GUST_FLAGS)}; {CLOCK}; grey = no data that day", top=0.88)
plt.show()

## 1b. Coverage, denominators and the flag filter (M4 2.4)

**Question:** what are the gust numbers made of: how many hours exist per airport and per local day,
against the 24 that could exist, and which of them can an aggregate trust?

**Coverage rule for every aggregate that covers a period.** A **local-day mean gust** needs at least
`MIN_SHARE` x 24 = 18 hours with a trusted gust (`MIN_SHARE = 0.75`). Reason: gusts have a daily cycle
(calm nights, windy afternoons; I-11 shows it at ESB), so a mean over only some hours describes
those hours and not the day. Six missing hours is the most that still leaves the day recognisable.
The windy-versus-calm comparison in section 4 has no partial-period aggregate: its unit is one
snapshot at one airport with one gust hour, and days are only the resampling unit (their number is
printed with every interval).

Clock: local time (`Europe/Istanbul`, converted with `tz_convert`), because the rule is about a day
as people live it. Every query below names its flag filter; both filters are shown.

In [ ]:
FILTERS = {"ok + unusual_high": GUST_FLAGS, "ok only": GUST_FLAGS_STRICT}
wf = {}
for name, flags in FILTERS.items():
    f = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(flags))
    f["local_date"] = explore.to_local(f.observed_at).dt.strftime("%Y-%m-%d")
    wf[name] = f

t = pd.to_datetime(w.observed_at, utc=True)
possible = int((t.max() - t.min()) / pd.Timedelta(hours=1)) + 1
stored = raw_vs_clean[raw_vs_clean.step == "1 raw, all hours"].set_index("iata").n_hours.drop("ALL")
den = pd.DataFrame({
    "possible (first to last observed hour)": possible,
    "stored in M2": stored,
    "observed (forecasts removed)": w.groupby("iata").size(),
    "trusted gust, ok": wf["ok only"].groupby("iata").gust_usable.sum(),
    "trusted gust, ok + unusual_high": wf["ok + unusual_high"].groupby("iata").gust_usable.sum(),
}).astype(int)
print("Hours per airport against the hours that could exist (UTC hours, first to last observed hour)")
display(den)
print("... as a share of the possible hours, %")
display((den.drop(columns=den.columns[0]).div(possible) * 100).round(0).astype(int))

In [ ]:
for name, f in wf.items():
    cov = explore.coverage_table(f, ["iata", "local_date"], "wind_gusts_10m", 24, MIN_SHARE)
    thin = cov[~cov.meets_rule].pivot(index="local_date", columns="iata", values="n").reindex(columns=airports)
    print(f"Local days below the {MIN_SHARE:.0%} rule (hours with a trusted gust out of 24); flag filter: {name}; "
          f"{cov.local_date.nunique()} local days with data per airport")
    display(thin.astype("Int64"))

In [ ]:
def daily_gust(f):
    d = f.groupby(["iata", "local_date"]).wind_gusts_10m.agg(n="count", mean="mean").reset_index()
    d["meets_rule"] = d.n >= MIN_SHARE * 24
    return d

rows = []
for name, f in wf.items():
    d = daily_gust(f)
    for iata, x in d.groupby("iata"):
        k = x[x.meets_rule]
        rows.append({"flag filter": name, "airport": iata, "days": len(x), "days kept by rule": len(k),
                     "mean of daily means, no rule": x["mean"].mean(), "mean of daily means, with rule": k["mean"].mean(),
                     "max daily mean, no rule": x["mean"].max(), "max daily mean, with rule": k["mean"].max(),
                     f"days with mean >= {WINDY_DAY_MEAN:g}, no rule": int((x["mean"] >= WINDY_DAY_MEAN).sum()),
                     f"days with mean >= {WINDY_DAY_MEAN:g}, with rule": int((k["mean"] >= WINDY_DAY_MEAN).sum())})
with_without = pd.DataFrame(rows).set_index(["flag filter", "airport"])
print("Local-day mean gust (km/h), without and with the 18-of-24-hours rule; n = days per airport")
display(with_without.round(2))

chg = (with_without["max daily mean, no rule"] - with_without["max daily mean, with rule"])
(fl, ap) = chg.idxmax()
print(f"- The rule drops {int((with_without.days - with_without['days kept by rule']).sum())} of {int(with_without.days.sum())} "
      f"airport-days over both filters; filter ok + unusual_high loses "
      f"{int((with_without.loc['ok + unusual_high'].days - with_without.loc['ok + unusual_high']['days kept by rule']).sum())} of "
      f"{int(with_without.loc['ok + unusual_high'].days.sum())}, filter ok only "
      f"{int((with_without.loc['ok only'].days - with_without.loc['ok only']['days kept by rule']).sum())} (the filter itself removes hours).")
print(f"- Largest effect on a result: the highest daily mean at {ap} under '{fl}' is "
      f"{with_without.loc[(fl, ap), 'max daily mean, no rule']:.1f} km/h without the rule and "
      f"{with_without.loc[(fl, ap), 'max daily mean, with rule']:.1f} km/h with it: a day built from few hours is not a day.")

## 2. Traffic grid: how much was airborne near each airport?

Query `02` gives **one row per snapshot and airport, zeros included**. A state in the overlap of
two airport areas is counted once, for the nearer airport. Before using it we check that the
grid has as many snapshots as there are landed OpenSky files: a snapshot whose states were all
repeats of earlier ones would be missing from the clean table, and with it its zeros.

In [ ]:
traffic = explore.query_df(con, "02_snapshot_traffic", TRAFFIC_PARAMS)
traffic["local_time"] = explore.to_local(traffic.snapshot)
traffic["local_date"] = traffic.local_time.dt.strftime("%Y-%m-%d")
traffic["local_hour"] = traffic.local_time.dt.hour + traffic.local_time.dt.minute / 60

n_snap = traffic.snapshot.nunique()
raw_dir = ROOT / "data" / "raw" / "opensky"
raw_files = sorted(raw_dir.glob("*.json")) if raw_dir.exists() else []
if raw_files:
    status = "OK" if len(raw_files) == n_snap else "MISMATCH: check for snapshots lost in M2/M3 deduplication"
    print(f"snapshots in grid: {n_snap}; landed OpenSky files: {len(raw_files)} -> {status}")
else:
    print(f"snapshots in grid: {n_snap}; raw files not available here, snapshot completeness not checked")

print(f"{len(traffic)} grid rows = {n_snap} snapshots x {traffic.iata.nunique()} airports; "
      f"{(traffic.n_aircraft == 0).sum()} rows are real zeros")
print(f"local days with at least one snapshot: {traffic.local_date.nunique()}")
per_airport = traffic.groupby("iata").agg(
    snapshots=("snapshot", "size"), mean_airborne=("n_airborne", "mean"),
    max_airborne=("n_airborne", "max"), mean_ground=("n_ground", "mean"))
per_airport.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
air2 = sorted(traffic.iata.unique())
col2 = dict(zip(air2, plt.cm.tab10.colors))
DODGE = 0.12                                   # hours; airports at one snapshot are shifted apart so no point hides another
for k_, iata in enumerate(air2):
    part_ = traffic[traffic.iata == iata]
    ax.scatter(part_.local_hour + (k_ - (len(air2) - 1) / 2) * DODGE, part_.n_airborne, color=col2[iata], alpha=0.85, label=iata)
    ax.axhline(part_.n_airborne.mean(), color=col2[iata], lw=1, ls="--", alpha=0.8)
ax.set_xlim(10, 24)
ax.set_ylim(-0.8, traffic.n_airborne.max() + 1)
ax.set_xlabel("local hour of the snapshot (Europe/Istanbul, UTC+3)")
ax.set_ylabel("airborne aircraft within 28 km of the airport")
ax.legend(title="airport (dashed line = its mean)", ncols=5, fontsize=8, loc="upper right")
means2 = traffic.groupby("iata").n_airborne.mean().sort_values()
fig.suptitle(f"Traffic is set by the airport, not the hour: {means2.index[-1]} averages {means2.iloc[-1]:.1f} aircraft near it, {means2.index[0]} {means2.iloc[0]:.1f}",
             fontsize=12, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.925, "Question: how much traffic is airborne near each airport, and at what times of day were snapshots taken?",
         fontsize=9.5, color="#444444")
explore.save_fig(fig, "fig02_traffic_by_local_hour",
                 f"n = {len(traffic)} snapshot-airport rows from {n_snap} snapshots; flag filter: aircraft positions with latitude_flag and "
                 f"longitude_flag = ok; radius {RADIUS_DEG} deg; {CLOCK}; points shifted sideways by up to {DODGE * 2:.2f} h so airports do not overlap",
                 top=0.89)
plt.show()

## 3. Joining traffic and weather

The join is a LEFT join from the traffic grid on `(airport, UTC hour)`; a snapshot at 10:59 uses
the weather of hour 10:00. Rows without a trusted gust stay in the table (`gust_usable = 0`) and
are counted below, never dropped silently.

In [ ]:
joined = explore.join_traffic_weather(traffic, w.drop(columns=["local_time", "local_date"]))
usable = joined[joined.gust_usable == 1]
print(f"{len(joined)} grid rows; {int(joined.has_weather.sum())} have a weather hour; "
      f"{len(usable)} have a trusted gust ({len(usable) / len(joined):.0%})")
print("rows lost to missing or untrusted gusts, by airport:")
print((joined.assign(lost=joined.gust_usable == 0).groupby("iata").lost.sum()).to_string())

## 4. Windy versus calm

For each airport the cut-off is its own 75th percentile of trusted gusts (median as a check).
The outcome is the airborne count and the mean airborne speed **minus the airport's mean**, so
Istanbul's larger traffic does not masquerade as a wind effect. The anomaly is computed on all
grid rows **before** rows are filtered by weather. The interval comes from resampling whole
local days (`bootstrap_diff`), and every line states how many days stand behind it.

In [ ]:
anom = explore.add_airport_anomaly(joined, ["n_airborne", "mean_velocity_airborne"])
OUTCOMES = {"n_airborne_anom": "airborne aircraft (vs airport mean)",
            "mean_velocity_airborne_anom": "mean airborne speed, m/s (vs airport mean)"}

def windy_calm(frame, q):
    cut = explore.gust_cutoffs(w, q)
    labelled = explore.split_by_threshold(frame, "wind_gusts_10m", cut)
    return labelled.dropna(subset=["group"]), cut

rows = []
for q in (WINDY_Q, WINDY_Q_ALT):
    part, cut = windy_calm(anom, q)
    counts = part.group.value_counts()
    for outcome in OUTCOMES:
        result = explore.bootstrap_diff(part, "local_date", outcome, "group", n_boot=N_BOOT, seed=SEED)
        rows.append({"windy_quantile": q, "outcome": outcome,
                     "n_calm": int(counts.get("calm", 0)), "n_windy": int(counts.get("windy", 0)),
                     "diff_windy_minus_calm": result[0], "lo": result[1], "hi": result[2],
                     "n_days": result[3]})
        print(f"q={q:.2f}  {OUTCOMES[outcome]}: {explore.format_diff(result, MIN_UNITS)}")
effect = pd.DataFrame(rows)
effect.round(2)

In [ ]:
# Does one airport drive the result? Repeat the main comparison leaving each airport out.
part, _ = windy_calm(anom, WINDY_Q)
loo = []
for iata in sorted(part.iata.unique()):
    sub = part[part.iata != iata]
    if sub.group.nunique() < 2:
        continue
    d, lo, hi, n = explore.bootstrap_diff(sub, "local_date", "n_airborne_anom", "group",
                                          n_boot=N_BOOT, seed=SEED)
    loo.append({"left_out": iata, "diff": d, "lo": lo, "hi": hi, "n_days": n})
loo = pd.DataFrame(loo)
loo.round(2)

In [ ]:
part, cut = windy_calm(anom, WINDY_Q)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True)
colors = dict(zip(sorted(part.iata.unique()), plt.cm.tab10.colors))
for ax, outcome in zip(axes, OUTCOMES):
    for iata, p in part.groupby("iata"):
        ax.scatter(p.wind_gusts_10m, p[outcome], color=colors[iata], label=iata, alpha=0.8)
    ax.axhline(0, color="#888888", lw=0.8)
    ax.set_xlabel("wind gust at the airport, km/h")
    ax.set_ylabel(OUTCOMES[outcome])
    sub = part.dropna(subset=[outcome])
    ax.set_title(f"Spearman {explore.correlation(sub.wind_gusts_10m, sub[outcome], 'spearman'):+.2f}, "
                 f"Pearson {explore.correlation(sub.wind_gusts_10m, sub[outcome]):+.2f}, n = {len(sub)} rows", fontsize=9)
axes[0].legend(title="airport", fontsize=8)
fig.suptitle("Stronger gusts do not visibly go with more or faster traffic: both clouds are flat",
             fontsize=12, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.925, "Question: do stronger gusts go together with more or faster airborne traffic near an airport?",
         fontsize=9.5, color="#444444")
explore.save_fig(fig, "fig03_traffic_vs_gust",
                 f"n = {len(part)} snapshot-airport rows with a trusted gust; flag filter: {', '.join(GUST_FLAGS)}; {CLOCK}; "
                 "0 line = the airport's own mean", top=0.9)
plt.show()

## 3b. Relationship: gusts and traffic (M4 2.6)

**Question:** among the snapshot-airport rows with a trusted gust, do stronger gusts go with more
airborne aircraft, or with faster ones (each relative to the airport's own mean)?

The scatter plots in `fig03` come first (above); the coefficients are computed only after looking at
them. **Pearson** measures a straight-line pattern and is moved by a few extreme points; **Spearman**
asks whether higher gusts tend to come with higher outcomes and ignores how extreme a point is. With
small integer counts and right-skewed gusts, Spearman describes the cloud better, so both are shown.
The rows come from a few snapshots on 10 local days and rows of one snapshot are not independent, so
every interval resamples local days. Flag filter: `ok` + `unusual_high`; clock: local time.

In [ ]:
BAND_LABELS = [f"{int(a):02d}-{int(b):02d} h" for a, b in zip(TIME_BAND_EDGES[:-1], TIME_BAND_EDGES[1:])]
u = anom[anom.gust_usable == 1].copy()
u["band"] = pd.cut(u.local_hour, TIME_BAND_EDGES, right=False, labels=BAND_LABELS)

COEF_OUTCOMES = {"n_airborne_anom": "airborne aircraft (vs airport mean)",
                 "mean_velocity_airborne_anom": "mean airborne speed, m/s (vs airport mean)"}
rows = []
for outcome, label in COEF_OUTCOMES.items():
    v = u.dropna(subset=[outcome])
    for method in ("pearson", "spearman"):
        r, lo, hi, n_days = explore.bootstrap_corr(v, "local_date", "wind_gusts_10m", outcome, method, N_BOOT, SEED)
        rows.append({"outcome": label, "method": method, "rows": len(v), "days": n_days, "r": r, "lo": lo, "hi": hi})
coef = pd.DataFrame(rows)
print(f"Correlation of the hourly gust with each outcome; flag filter: {', '.join(GUST_FLAGS)}; interval resamples local days")
display(coef.round(2))

In [ ]:
def pair(frame, x="wind_gusts_10m", y="n_airborne_anom"):
    return {"rows": len(frame), "Pearson": explore.correlation(frame[x], frame[y]),
            "Spearman": explore.correlation(frame[x], frame[y], "spearman")}

storm_day = "2026-10-27"
by_band = u.groupby("band", observed=True)
by_snap = u.groupby("snapshot")
centred_band = u.assign(g=u.wind_gusts_10m - by_band.wind_gusts_10m.transform("mean"),
                        y=u.n_airborne_anom - by_band.n_airborne_anom.transform("mean"))
centred_snap = u.assign(g=u.wind_gusts_10m - by_snap.wind_gusts_10m.transform("mean"),
                        y=u.n_airborne_anom - by_snap.n_airborne_anom.transform("mean"))
u["gust_airport_centred"] = u.wind_gusts_10m - u.groupby("iata").wind_gusts_10m.transform("mean")
drivers = pd.DataFrame({
    "all rows": pair(u),
    f"without {storm_day} (storm day at IST)": pair(u[u.local_date != storm_day]),
    "without the 3 highest gusts": pair(u.nsmallest(len(u) - 3, "wind_gusts_10m")),
    "without the top 5% of gusts": pair(u.nsmallest(int(len(u) * 0.95), "wind_gusts_10m")),
    "gust centred by airport": pair(u, "gust_airport_centred"),
    "both centred within time-of-day band": pair(centred_band, "g", "y"),
    "both centred within snapshot": pair(centred_snap, "g", "y"),
}).T
drivers["rows"] = drivers.rows.astype(int)
print("Airborne-count relationship: what drives it? (one event / shared pattern in group or time)")
display(drivers.round(2))
per_airport = pd.DataFrame({i: pair(x) for i, x in u.groupby("iata")}).T
per_airport["rows"] = per_airport.rows.astype(int)
print("... within each airport (n = rows; one airport at a time)")
display(per_airport.round(2))

In [ ]:
vel = u.dropna(subset=["mean_velocity_airborne_anom"])
vel_rule = vel[vel.n_airborne >= MIN_AIRBORNE_SPEED]
speed = pd.DataFrame({
    "no rule": pair(vel, y="mean_velocity_airborne_anom"),
    f"only snapshots with >= {MIN_AIRBORNE_SPEED} airborne aircraft": pair(vel_rule, y="mean_velocity_airborne_anom"),
}).T
speed["rows"] = speed.rows.astype(int)
print(f"Speed relationship with and without a coverage rule (a mean speed needs >= {MIN_AIRBORNE_SPEED} aircraft)")
display(speed.round(2))
looks_26 = len(coef) + 2 * len(drivers) + 2 * len(per_airport) + 2 * len(speed)

cn = coef[coef.outcome.str.startswith("airborne")].set_index("method")
cv = coef[coef.outcome.str.startswith("mean airborne")].set_index("method")
def span(row):
    return f"[{row.lo:+.2f}, {row.hi:+.2f}]"
print(f"- Count: Pearson {cn.loc['pearson', 'r']:+.2f} {span(cn.loc['pearson'])}, Spearman {cn.loc['spearman', 'r']:+.2f} "
      f"{span(cn.loc['spearman'])} (n = {int(cn.rows.iloc[0])} rows, {int(cn.days.iloc[0])} days). Speed: Pearson "
      f"{cv.loc['pearson', 'r']:+.2f}, Spearman {cv.loc['spearman', 'r']:+.2f} (n = {int(cv.rows.iloc[0])} rows).")
print("- Intervals include 0 for", "all four coefficients." if bool(((coef.lo <= 0) & (coef.hi >= 0)).all())
      else "some, not all, coefficients: " + ", ".join(f"{r_.outcome[:12]} {r_.method}" for r_ in coef.itertuples() if not r_.lo <= 0 <= r_.hi))
storm_rows = u[u.local_date == storm_day]
ist_storm = w[(w.iata == "IST") & (w.local_date == storm_day)].wind_gusts_10m.max()
print(f"- One event? {len(storm_rows)} rows fall on {storm_day}, with gusts of at most {storm_rows.wind_gusts_10m.max():.1f} km/h, "
      f"while IST reached {ist_storm:.1f} km/h that day: the storm hours fell between snapshots. Without that day the Spearman "
      f"value is {drivers.iloc[1].Spearman:+.2f}; without the top 5% of gusts, {drivers.iloc[3].Spearman:+.2f}.")
print(f"- Shared patterns? Centring within time-of-day band gives Spearman {drivers.iloc[5].Spearman:+.2f}, within snapshot "
      f"{drivers.iloc[6].Spearman:+.2f}; by airport it runs from {per_airport.Spearman.min():+.2f} ({per_airport.Spearman.idxmin()}) "
      f"to {per_airport.Spearman.max():+.2f} ({per_airport.Spearman.idxmax()}) with only {int(per_airport.rows.min())} to "
      f"{int(per_airport.rows.max())} rows each.")
n_small = int((vel.n_airborne < MIN_AIRBORNE_SPEED).sum())
print(f"- Speed: {n_small} of {len(vel)} speed values rest on fewer than {MIN_AIRBORNE_SPEED} aircraft; with the rule Pearson is "
      f"{speed.iloc[1].Pearson:+.2f} and Spearman {speed.iloc[1].Spearman:+.2f} (n = {int(speed.iloc[1].rows)}).")

**Reading.** The clouds in `fig03` are flat and the coefficients are small: there is no usable
relationship between gusts and airborne traffic in these snapshots. Spearman is the better
description for the count outcome (small integers with many ties, skewed gusts). For speed, Pearson is
moved by a few extreme values that rest on one or two aircraft, so a mean speed needs a minimum number of
aircraft (`MIN_AIRBORNE_SPEED`). The one event (27 Oct) is not driving anything: no snapshot caught its
strong hours. What the airport-by-airport rows hint at is a **hypothesis for a larger sample**, not a
result, since each airport has only 15 or 16 rows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, outcome in zip(axes, OUTCOMES):
    sub = effect[effect.outcome == outcome].reset_index(drop=True)
    y = np.arange(len(sub))
    ax.errorbar(sub.diff_windy_minus_calm, y,
                xerr=[sub.diff_windy_minus_calm - sub.lo, sub.hi - sub.diff_windy_minus_calm],
                fmt="o", capsize=4, color="#1f77b4")
    ax.axvline(0, color="#888888", lw=0.8)
    ax.set_yticks(y, [f"windy = above airport {int(q * 100)}th pct" for q in sub.windy_quantile])
    ax.set_xlabel("windy minus calm, 95% interval")
    ax.set_title(OUTCOMES[outcome], fontsize=9)
all_zero = bool(((effect.lo <= 0) & (effect.hi >= 0)).all())
fig.suptitle("Windy hours show no detectable traffic difference: every interval includes 0" if all_zero else
             "Windy versus calm: at least one interval excludes 0 (exploratory)",
             fontsize=12, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.925, f"Question: is airborne traffic different when gusts are windy, over {int(effect.n_days.max())} local days?",
         fontsize=9.5, color="#444444")
explore.save_fig(fig, "fig04_windy_vs_calm",
                 f"n = {int(effect.n_days.max())} local days; bootstrap of days, {N_BOOT} draws, seed {SEED}; "
                 f"flag filter: {', '.join(GUST_FLAGS)}; {CLOCK}", top=0.9)
plt.show()

## 4b. The key comparison under two flag filters (M4 2.4)

The same windy-versus-calm comparison as in section 4 (reported filter `ok` + `unusual_high`;
outcome: airborne aircraft versus the airport's mean; unit: local day), repeated with the strict
filter `ok` only. The airport cut-off for "windy" is recomputed from the trusted hours of each
filter, because the filter changes which hours exist. Same snapshots, same seed, same bootstrap.

In [ ]:
def windy_calm_effect(flags, q=WINDY_Q):
    wfl = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(flags))
    jn = explore.join_traffic_weather(traffic, wfl)
    an = explore.add_airport_anomaly(jn, ["n_airborne", "mean_velocity_airborne"])
    cut = explore.gust_cutoffs(wfl, q)
    part = explore.split_by_threshold(an, "wind_gusts_10m", cut).dropna(subset=["group"])
    counts = part.group.value_counts()
    out = []
    for outcome in OUTCOMES:
        d, lo, hi, n = explore.bootstrap_diff(part, "local_date", outcome, "group", n_boot=N_BOOT, seed=SEED)
        out.append({"outcome": OUTCOMES[outcome], "usable_rows": int((an.gust_usable == 1).sum()),
                    "n_calm": int(counts.get("calm", 0)), "n_windy": int(counts.get("windy", 0)),
                    "diff": d, "lo": lo, "hi": hi, "n_days": int(n), "text": explore.format_diff((d, lo, hi, n), MIN_UNITS)})
    return pd.DataFrame(out), cut

two = {}
for name, flags in {"ok + unusual_high (reported)": GUST_FLAGS, "ok only (strict)": GUST_FLAGS_STRICT}.items():
    two[name], cut = windy_calm_effect(flags)
    print(f"flag filter: {name}; windy cut-offs (km/h, airport {WINDY_Q:.0%} quantile): "
          + ", ".join(f"{k} {v:.1f}" for k, v in sorted(cut.items())))
    for r in two[name].itertuples():
        print(f"  {r.outcome}: {r.text}; {r.n_windy} windy vs {r.n_calm} calm rows of {r.usable_rows} with a trusted gust")
same = all((r.lo <= 0 <= r.hi) for t_ in two.values() for r in t_.itertuples())
print("- Both filters give the same reading: the interval includes 0 for every outcome." if same else
      "- The filters disagree: at least one interval excludes 0. Report both.")

**Which filter we report: `ok` + `unusual_high`.** Three reasons. `unusual_high` hours are real
values that M3 kept unchanged and only flagged (I-11, I-12). The strict filter removes the top of
each airport's distribution, which is the part "windy" is about (0b: the tail ranking changes), and it
thins ESB and AYT so that only 9 and 11 of 16 local days pass the 18-of-24 rule (1b). And the reading of
the comparison is the same under both filters (printed line above), so this choice does not drive the
result. The strict filter stays as the sensitivity check.

## 4c. Time (M4 2.5)

**Question:** which clock do the time-based numbers use, are the first and last days of the data
complete in that clock, how do gusts and traffic vary over the day and across days, and do
holidays touch any of it?

**Clock: local time, `Europe/Istanbul`.** Flights and gusts are about the day as it is lived (the
afternoon, a Tuesday, a holiday), and the stored UTC hour sits three hours earlier. Every time
grouping below converts explicitly with `tz_convert` (`explore.to_local`); no UTC hour is grouped for
a question about the day. Turkey has no daylight saving time, so the offset is a constant +3 h; the
first cell below checks that in the data instead of assuming it. Flag filter: `ok` + `unusual_high`;
forecast hours excluded.

In [ ]:
tw_all = w.copy()                                   # all observed hours, local_time already converted in section 1
offsets = sorted({str(x.utcoffset()) for x in tw_all.local_time})
print("UTC offsets present in the converted local times:", offsets)

per_day = (tw_all.groupby(["local_date", "iata"]).size().unstack())
first_day, last_day = per_day.index.min(), per_day.index.max()
edge_rows = []
for label, day in (("first local day", first_day), ("last local day", last_day)):
    hrs = tw_all[tw_all.local_date == day].local_time
    edge_rows.append({"edge": label, "local_date": day, "first hour": hrs.min().strftime("%H:%M"),
                      "last hour": hrs.max().strftime("%H:%M"), "hours per airport (min)": int(per_day.loc[day].min()),
                      "meets 18-of-24 rule": bool(per_day.loc[day].min() >= MIN_SHARE * 24)})
snap = traffic.groupby("snapshot").local_time.first().sort_values()
edge_rows.append({"edge": "first / last ADS-B snapshot", "local_date": f"{snap.iloc[0]:%Y-%m-%d} / {snap.iloc[-1]:%Y-%m-%d}",
                  "first hour": f"{snap.iloc[0]:%H:%M}", "last hour": f"{snap.iloc[-1]:%H:%M}",
                  "hours per airport (min)": None, "meets 18-of-24 rule": None})
edges = pd.DataFrame(edge_rows).set_index("edge")
print("Edges of the data in local time (weather: hours with data that local day)")
display(edges)
utc_first = tw_all.observed_at.str[:10].min()
print(f"- In UTC the first weather day ({utc_first}) would look complete with 24 hours; in local time it starts at "
      f"{edges.loc['first local day', 'first hour']} and has {int(edges.loc['first local day', 'hours per airport (min)'])} hours.")
print(f"- The last local day ends at {edges.loc['last local day', 'last hour']} with "
      f"{int(edges.loc['last local day', 'hours per airport (min)'])} hours: it "
      f"{'passes' if edges.loc['last local day', 'meets 18-of-24 rule'] else 'fails'} the 18-of-24 rule, so its daily mean is not used.")
clock_of_day = snap.dt.hour + snap.dt.minute / 60
print(f"- ADS-B snapshots were taken between {snap[clock_of_day.idxmin() == snap.index].iloc[0]:%H:%M} and "
      f"{snap[clock_of_day.idxmax() == snap.index].iloc[0]:%H:%M} local time only "
      f"({len(snap)} snapshots on {snap.dt.strftime('%Y-%m-%d').nunique()} local days): never at night or in the morning.")

In [ ]:
tg = w[w.gust_usable == 1].copy()
tg["hour"] = tg.local_time.dt.hour
prof = tg.groupby(["iata", "hour"]).wind_gusts_10m.agg(mean="mean", n="count").reset_index()
daily = daily_gust(wf["ok + unusual_high"])
full_dates = pd.date_range(daily.local_date.min(), daily.local_date.max(), freq="D")
dec_hour = snap.dt.hour + snap.dt.minute / 60
snap_lo, snap_hi = float(dec_hour.min()), float(dec_hour.max())

fig, (ax_h, ax_d) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1.9]}, sharey=True)
ax_h.axvspan(snap_lo, snap_hi, color="#dddddd", alpha=0.6, lw=0, label="window of the ADS-B snapshots")
for iata in airports:
    pr = prof[prof.iata == iata]
    ax_h.plot(pr.hour, pr["mean"], color=colors[iata], marker="o", ms=3, label=iata)
ax_h.set_xticks(range(0, 24, 3))
ax_h.set_xlim(0, 23)
ax_h.set_xlabel("local hour of day (Europe/Istanbul, UTC+3)")
ax_h.set_ylabel("mean wind gust at 10 m, km/h")
ax_h.set_title(f"By hour of day (n = {int(prof.n.min())} to {int(prof.n.max())} days per point)", fontsize=10)
ax_h.legend(fontsize=8, ncols=2, loc="upper left")

for d_, name in HOLIDAYS.items():
    ax_d.axvspan(pd.Timestamp(d_), pd.Timestamp(d_) + pd.Timedelta(days=1), color="#c44e52", alpha=0.18, lw=0)
for d_ in full_dates[full_dates.dayofweek >= 5]:
    ax_d.axvspan(d_, d_ + pd.Timedelta(days=1), color="#999999", alpha=0.12, lw=0)
for iata in airports:
    dd = daily[daily.iata == iata].assign(date=lambda x: pd.to_datetime(x.local_date)).set_index("date")
    kept = dd[dd.meets_rule]["mean"].reindex(full_dates)             # missing days stay NaN: the line breaks
    ax_d.plot(full_dates, kept, color=colors[iata], marker="o", ms=3.5, lw=1.2)
    thin = dd[~dd.meets_rule]
    ax_d.scatter(thin.index, thin["mean"], facecolors="white", edgecolors=colors[iata], s=22, zorder=3)
ax_d.axhline(WINDY_DAY_MEAN, color="black", lw=0.8, ls="--")
ax_d.text(full_dates[0], WINDY_DAY_MEAN + 0.7, f"{WINDY_DAY_MEAN:g} km/h", fontsize=8)
ax_d.set_xlim(full_dates[0] - pd.Timedelta(hours=12), full_dates[-1] + pd.Timedelta(hours=36))
ax_d.set_xticks(full_dates[::3])
ax_d.set_xticklabels([d_.strftime("%m-%d") for d_ in full_dates[::3]], fontsize=8)
ax_d.set_xlabel("local date, 2026 (Europe/Istanbul)")
ax_d.set_title(f"By day (n = {int(daily.meets_rule.sum())} days kept of {len(daily)} airport-days)", fontsize=10)
ax_d.plot([], [], "o", mfc="white", mec="black", ms=5, ls="", label=f"fewer than {int(MIN_SHARE * 24)} of 24 hours (not joined)")
ax_d.fill_between([], [], color="#c44e52", alpha=0.3, label="public holiday (28-29 Oct): no data")
ax_d.fill_between([], [], color="#999999", alpha=0.25, label="weekend")
ax_d.legend(fontsize=8, loc="upper right")
ax_h.set_ylim(0, 45)

fig.suptitle("Gusts follow the clock and the weather: an afternoon peak at every airport, one storm day at Istanbul",
             fontsize=13, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.935, "Question: how do gusts vary over the day and across days, and does the clock or a holiday explain it?",
         fontsize=10, color="#444444")
explore.save_fig(fig, "fig06_time_scales",
                 f"n = {len(tg)} hours with a trusted gust ({tg.local_date.nunique()} local days x {len(airports)} airports); "
                 f"flag filter: {', '.join(GUST_FLAGS)}; forecast hours excluded; clock: Europe/Istanbul via tz_convert; "
                 f"daily means need >= {int(MIN_SHARE * 24)} of 24 hours",
                 top=0.92)
plt.show()

In [ ]:
pm = prof.pivot(index="hour", columns="iata", values="mean")
peak = pm.idxmax()
ratio = pm.loc[12:17].mean() / pm.loc[0:5].mean()
print("Hour of day (local): peak hour and afternoon (12-17) over night (0-5) mean gust")
display(pd.DataFrame({"peak hour": peak, "afternoon / night": ratio.round(2)}))
dm = daily[daily.meets_rule].groupby("iata")["mean"].agg(["min", "max"])
dm["range of daily means"] = dm["max"] - dm["min"]
dm["range of hourly means"] = pm.max() - pm.min()
print("Which scale moves the gusts more: across days or across the day? (km/h; n = days kept per airport)")
display(dm.assign(n_days=daily[daily.meets_rule].groupby("iata").size()).round(1))

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weather_days = pd.Series(sorted(w.local_date.unique()))
snap_days = pd.Series(sorted(snap.dt.strftime("%Y-%m-%d").unique()))
dow = pd.DataFrame({
    "weather days": pd.to_datetime(weather_days).dt.day_name().value_counts(),
    "days with snapshots": pd.to_datetime(snap_days).dt.day_name().value_counts()}).reindex(dow_order).fillna(0).astype(int)
print("Day of week: local days in the data (n per weekday is 0 to 3, too thin to compare weekdays)")
display(dow)

hol = pd.DataFrame({
    "holiday": pd.Series(HOLIDAYS),
    "weather hours": [int((w.local_date == d_).sum()) for d_ in HOLIDAYS],
    "snapshots": [int((snap.dt.strftime("%Y-%m-%d") == d_).sum()) for d_ in HOLIDAYS]})
print("Calendar: Turkish public holidays inside the span, and how much data falls on them")
display(hol)
weekend_snap = int((snap.dt.dayofweek >= 5).sum())
print(f"- {int(hol['weather hours'].sum())} weather hours and {int(hol.snapshots.sum())} snapshots fall on the holiday and its eve: "
      "the holiday neither contaminates nor can be compared; it sits inside the gap between 27 and 30 Oct (I-18).")
print(f"- Weekends: {weekend_snap} of {len(snap)} snapshots and {int((pd.to_datetime(weather_days).dt.dayofweek >= 5).sum())} of "
      f"{len(weather_days)} weather days are Saturday or Sunday; traffic can differ on weekends, which matters for the windy-versus-calm mix (2.7).")

**What the time view shows.**

- *Clock.* Gusts peak between 15:00 and 18:00 local time at every airport, and the afternoon mean is
  1.3 to 3.2 times the night mean (Ankara the most). In UTC that peak would sit at 12:00 to 15:00 and be
  read as noon.
- *Two scales.* Across the day, the cycle is of the same size as the movement between days at ADB, AYT and
  ESB; at IST and SAW the days differ more than the hours do, because of one storm day (27 Oct).
  Gusts are therefore not a pure weather signal: the time of day alone moves them by 8 to 13 km/h.
- *Edges.* The first local day starts at 03:00 and the last ends at 11:00: with the rule the last one
  is dropped; the first stays (21 of 24 hours).
- *Calendar.* The Republic Day holiday (29 Oct) and its eve fall in a gap with no data, so no holiday
  is in the numbers. Day-of-week comparisons are not possible with 0 to 3 days per weekday.
- *What it means for the traffic comparison.* All ADS-B snapshots were taken between late morning
  and late evening local time, spanning the afternoon gust peak, and none at night. Time of day moves both
  gusts and traffic, so it confounds the windy-versus-calm comparison; the composition check in 2.7
  must compare within time-of-day bands.

## 4d. Composition and the number of looks (M4 2.7)

**The comparison.** Airborne aircraft near the airport (relative to its mean), windy minus calm,
where windy means a gust at or above the airport's own 75th percentile; difference and 95% interval
are in section 4 (and 4b for the second filter). The unit is the local day, and the interval resamples
days, not rows.

**Composition check.** The windy and calm groups must be made of the same mixture before a difference
can be read: the mix by airport, by time of day (the gust peak is in the afternoon, 4c) and by weekday
or weekend. If they differ, the comparison is repeated within subgroups to see whether it reverses
(Simpson's paradox).

In [ ]:
part, cut = windy_calm(anom, WINDY_Q)
part = part.copy()
part["band"] = pd.cut(part.local_hour, TIME_BAND_EDGES, right=False, labels=BAND_LABELS)
part["weekend"] = np.where(pd.to_datetime(part.local_date).dt.dayofweek >= 5, "weekend", "weekday")
n_calm, n_windy = int((part.group == "calm").sum()), int((part.group == "windy").sum())

def mixture(col):
    ct = pd.crosstab(part[col], part.group)[["calm", "windy"]]
    return pd.concat({"rows": ct, "share of group, %": (ct / ct.sum() * 100).round(0).astype(int)}, axis=1)

for col, title in (("iata", "airport"), ("band", "local hour of the snapshot"), ("weekend", "day type")):
    print(f"Mixture of the groups by {title} (windy = above the airport's {int(WINDY_Q * 100)}th percentile; "
          f"flag filter: {', '.join(GUST_FLAGS)}; n = {n_calm} calm, {n_windy} windy rows)")
    display(mixture(col))
share_band = pd.crosstab(part.band, part.group, normalize="columns") * 100
big = (share_band.windy - share_band.calm).abs().idxmax()
print(f"- Time of day differs most: {share_band.loc[big, 'windy']:.0f}% of the windy rows but {share_band.loc[big, 'calm']:.0f}% "
      f"of the calm rows fall in {big}. The groups are not made of the same mixture.")

In [ ]:
part["adjusted"] = part.n_airborne_anom - part.groupby("band", observed=True).n_airborne_anom.transform("mean")
pooled = explore.bootstrap_diff(part, "local_date", "n_airborne_anom", "group", n_boot=N_BOOT, seed=SEED)
adjusted = explore.bootstrap_diff(part, "local_date", "adjusted", "group", n_boot=N_BOOT, seed=SEED)
cell = part.groupby(["band", "group"], observed=True).n_airborne_anom.agg(["mean", "count"]).unstack("group")
within = pd.DataFrame({"windy rows": cell[("count", "windy")], "calm rows": cell[("count", "calm")],
                       "windy minus calm": cell[("mean", "windy")] - cell[("mean", "calm")]})
print("Windy minus calm, airborne aircraft vs airport mean: pooled, adjusted for time-of-day band, and within each band")
print("  pooled:  ", explore.format_diff(pooled, MIN_UNITS))
print("  adjusted:", explore.format_diff(adjusted, MIN_UNITS))
display(within.round(2))
flipped = [b for b, d_ in within["windy minus calm"].items() if np.sign(d_) != np.sign(pooled[0])]
print(f"- Pooled {pooled[0]:+.2f} versus adjusted {adjusted[0]:+.2f}: adjusting for the time-of-day mix "
      f"{'does not change' if abs(adjusted[0] - pooled[0]) < 0.25 else 'changes'} the difference materially.")
print("- Within bands the sign " + (f"differs from the pooled sign in {', '.join(flipped)} " if flipped else "matches the pooled sign in every band ")
      + f"(cells of {int(within[['windy rows', 'calm rows']].min().min())} to {int(within[['windy rows', 'calm rows']].max().max())} rows): "
      + ("the pooled sign does not reverse, but the bands do not agree, so the difference is not a stable effect." if flipped else "no reversal."))

In [ ]:
looks = pd.DataFrame([
    ("windy cut-off 75th vs 50th percentile x 2 outcomes (section 4)", len(effect)),
    ("leave-one-airport-out (section 4)", len(loo)),
    ("second flag filter x 2 outcomes (4b)", int(sum(len(t_) for t_ in two.values()))),
    ("time bands, band-adjusted difference (4d)", len(within) + 1),
    ("correlations, driver checks, airports, speed rule (3b)", looks_26),
], columns=["looks at the data", "comparisons"]).set_index("looks at the data")
display(looks)
print(f"- Roughly {int(looks.comparisons.sum())} comparisons or coefficients were looked at in this notebook. The primary comparison "
      "(airborne count, windy = above the airport's 75th percentile) was fixed in the decisions table before any traffic outcome was "
      "seen; the others were looked at afterwards. With that many looks, one interval excluding 0 would be a hypothesis, not a finding.")
print("- Result:", "every interval for the primary comparison includes 0." if bool(((effect.lo <= 0) & (effect.hi >= 0)).all())
      else "at least one interval for the primary comparison excludes 0; treat it as exploratory.")

In [ ]:
fig, (ax_c, ax_d) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.15]})
xb = np.arange(len(share_band))
wd = 0.38
ax_c.bar(xb - wd / 2, share_band.calm, wd, color="#4c72b0", label=f"calm (n = {n_calm} rows)")
ax_c.bar(xb + wd / 2, share_band.windy, wd, color="#dd8452", label=f"windy (n = {n_windy} rows)")
ax_c.set_xticks(xb, share_band.index)
ax_c.set_xlabel("local hour of the snapshot (Europe/Istanbul)")
ax_c.set_ylabel("share of the group's rows, %")
ax_c.set_ylim(0, 70)
ax_c.set_title("The groups are not made of the same hours", fontsize=10)
ax_c.legend(fontsize=8)

labels_d = ["all rows, pooled\n(95% interval)", "adjusted for time band\n(95% interval)"] + [f"{b_}\n(too few rows for an interval)" for b_ in within.index]
ax_d.errorbar([pooled[0], adjusted[0]], [0, 1], xerr=[[pooled[0] - pooled[1], adjusted[0] - adjusted[1]],
                                                       [pooled[2] - pooled[0], adjusted[2] - adjusted[0]]],
              fmt="o", capsize=4, color="#1f77b4")
ax_d.plot(within["windy minus calm"], np.arange(2, 2 + len(within)), "o", mfc="white", mec="#555555", ls="")
for k, (b_, r_) in enumerate(within.iterrows()):
    ax_d.text(r_["windy minus calm"], 2 + k - 0.3, f"{int(r_['windy rows'])} windy / {int(r_['calm rows'])} calm", fontsize=7, ha="center")
ax_d.axvline(0, color="#888888", lw=0.8)
ax_d.set_yticks(range(len(labels_d)), labels_d)
ax_d.invert_yaxis()
ax_d.set_xlabel("windy minus calm, airborne aircraft vs airport mean")
ax_d.set_title(f"Adjusting for time of day ({pooled[0]:+.2f} to {adjusted[0]:+.2f}) does not create a difference", fontsize=10)
fig.suptitle("Windy snapshots cluster in the afternoon, but time of day does not explain or hide a traffic difference",
             fontsize=12, fontweight="bold", x=0.01, y=0.995, ha="left")
fig.text(0.01, 0.935, "Question: are the windy and calm groups made of the same mixture, and does the comparison survive within subgroups?",
         fontsize=9.5, color="#444444")
explore.save_fig(fig, "fig07_composition_windy_calm",
                 f"n = {len(part)} snapshot-airport rows with a trusted gust ({n_windy} windy, {n_calm} calm) on {part.local_date.nunique()} local days; "
                 f"flag filter: {', '.join(GUST_FLAGS)}; windy = above the airport's {int(WINDY_Q * 100)}th percentile; {CLOCK}; intervals resample local days", top=0.92)
plt.show()

**Reading.** The windy group is concentrated in the afternoon band, so the two groups are not made
of the same hours; adjusting for that leaves the difference where it was, and it stays compatible with
zero. The within-band differences disagree in sign, each from a handful of rows, so there is no stable
effect to report. The number of looks above is the denominator for any single striking value.

## 5. What we can say

The text below is generated from the results above, so it cannot drift from them.

In [ ]:
main = effect[effect.windy_quantile == WINDY_Q].set_index("outcome")
n_days = int(main.n_days.iloc[0])
for outcome, label in OUTCOMES.items():
    r = main.loc[outcome]
    excludes = not (r.lo <= 0 <= r.hi)
    print(f"- {label}: windy minus calm = {r.diff_windy_minus_calm:+.2f} "
          f"[{r.lo:+.2f}, {r.hi:+.2f}], {int(r.n_windy)} windy vs {int(r.n_calm)} calm rows, "
          f"{n_days} local days -> {'interval excludes 0' if excludes else 'no detectable difference'}.")
if n_days < MIN_UNITS:
    print(f"- Only {n_days} independent days: treat every line above as exploratory.")
print(f"- Snapshot timing is irregular ({n_snap} snapshots, {traffic.local_date.nunique()} local days): "
      "time of day is not controlled and confounds any wind effect.")
print("- Next step for a firmer answer: more snapshots on more days, spread over the day, "
      "and a model with time of day and airport as covariates.")

## 5b. Findings, honestly labelled (M4 2.8)

Numbers are from this run (flag filter `ok` + `unusual_high`, 16 weather days, 16 ADS-B snapshots on 10
local days, bootstrap of days with seed 20261103). *Exploratory* = seen in this data, not yet tested on
data we did not look at; *Confirmed* = holds by construction or by arithmetic. **"Explored by" is a
proposal: each member must be able to explain the rows with their name at M8; change it to match who did the work.**

| ID | Finding | Evidence | Label | What would change it | Explored by |
|---|---|---|---|---|---|
| F-01 | Hourly gusts are right-skewed at every airport (mean above median everywhere), so the typical hour is calm and a long tail carries the strong hours; we summarise the typical hour with the median and strong hours with the 90th percentile | `figures/fig05_gust_shape.png`; sections 0b, 0c | Exploratory | A different season or wind regime; a longer record than 16 days | Mehmetali Açık |
| F-02 | ADB has the higher median gust (11.5 vs 10.1 km/h) but AYT the higher 90th percentile (23.0 vs 21.2 km/h): the two summaries rank them differently, answering "typical hour" versus "strong hours" | Section 0b, rank tables | Exploratory | More days (the gap is small); a different tail definition | Mehmetali Açık |
| F-03 | The strict flag filter (`ok` only) cuts the top of each airport's distribution: ESB's 90th percentile falls from 18.7 to 11.9 km/h and only 9 of 16 ESB days (11 at AYT) pass the 18-of-24-hours rule, against 14 of 16 at every airport with `ok` + `unusual_high` | Section 1b tables; `figures/fig01_weather_coverage.png` | Confirmed (arithmetic) | A different `ROBUST_Z_THRESHOLD` or a per-hour-of-day baseline in `clean.py` (I-11) | Enes Akdeniz |
| F-04 | The raw mean gust of 14.69 km/h becomes 14.93 km/h in the cleaned layer (forecast hours -260, one stuck value NULLed); the strict filter would give 14.23 km/h and drop ESB's mean from 8.21 to 6.34 km/h | Section 0 (queries `05`, `06`) | Confirmed (arithmetic) | Nothing, unless the flag rules change | Enes Akdeniz |
| F-05 | Gusts peak between 15:00 and 18:00 local time at every airport; the afternoon mean is 1.3 to 3.2 times the night mean (Ankara the most). In UTC the peak would read as noon | `figures/fig06_time_scales.png`; section 4c | Exploratory | Night and morning data from other days; another season | Enes Akdeniz |
| F-06 | All 16 ADS-B snapshots were taken between 11:34 and 22:38 local time, so time of day confounds the windy-versus-calm comparison: 55% of the windy rows but 24% of the calm rows fall between 15:00 and 19:00 | `figures/fig07_composition_windy_calm.png`; section 4d | Confirmed (composition) | Snapshots spread over the whole day | Nihat Avcı |
| F-07 | The edge days and holidays are handled: 8 Oct (21 hours) passes the rule, 21 Oct (17) and 3 Nov (12) fail it, and Republic Day (28-29 Oct) falls in a gap with no data, so no holiday is in the numbers | Section 4c edge and calendar tables | Confirmed (arithmetic) | Data collected on a holiday | Enes Akdeniz |
| F-08 | No relationship between gusts and airborne traffic is visible: Spearman +0.14 [-0.22, +0.34] and Pearson +0.12 for the count, +0.01 and +0.12 for the mean speed (n = 79 and 66 rows, 10 days); the clouds in the scatter plots are flat | `figures/fig03_traffic_vs_gust.png`; section 3b | Exploratory | More snapshots on more days, spread over the day; a model with hour and airport | Nihat Avcı |
| F-09 | One event does not drive it: no snapshot caught the 27 Oct storm (snapshot gusts at most 16.6 km/h, IST peaked at 57.6 km/h), and removing that day moves Spearman from +0.14 to +0.16; centring by time-of-day band or by snapshot leaves +0.11 and +0.09 | Section 3b driver table | Confirmed (arithmetic) | A snapshot taken during a storm hour | Nihat Avcı |
| F-10 | Within airports the sign differs (ESB +0.52 and IST +0.41, ADB -0.34 and AYT -0.16 in Spearman), with only 15 or 16 rows each; this is a hypothesis for a larger sample, not a result | Section 3b per-airport table | Exploratory | More snapshots per airport | Mehmetali Açık |
| F-11 | Windy minus calm airborne count is +0.41 aircraft, 95% interval [-0.77, +1.80] over 10 days; it includes 0 under both flag filters, the band-adjusted value is +0.35, the sign flips in the 15-19 h band, and moving the windy cut-off to the median gives -0.53 [-1.82, +0.84]. About 49 comparisons and coefficients were looked at | `figures/fig04_windy_vs_calm.png`; sections 4, 4b, 4d | Exploratory | A third month of data; snapshots in every time band | Nihat Avcı |
| F-12 | A mean airborne speed rests on one or two aircraft in 20 of 66 snapshot-airport rows; with at least 3 aircraft (46 rows) the speed correlation is Pearson -0.07 and Spearman -0.20 instead of +0.12 and +0.01 | Section 3b speed table | Confirmed (arithmetic) for the 20 of 66; exploratory for the correlation | A larger minimum or a count-weighted mean | Mehmetali Açık |

### Back to cleaning

Nothing found in the aircraft or weather tables that the cleaned layer got wrong. Three items for the
M3 issues register (`docs/cleaning_report.md`) and, where marked, `clean.py`; **none is changed yet, the
team decides**:

1. **I-11 (`unusual_high` gusts).** Exploration showed the cost of filtering on this flag: it removes the
   top of every distribution and leaves ESB with 9 of 16 days. Add this as evidence to I-11 and consider a
   per-hour-of-day baseline or a higher `ROBUST_Z_THRESHOLD` in `clean.py`. Until then queries keep the flag
   and filter on it only as a sensitivity check.
2. **I-20 (forecast hours).** 260 of 1,920 weather rows are forecasts and the cleaned table does not
   flag them, so every query must remember `observed_at <= retrieved_at`. Consider an `is_forecast` column in `clean.py`
   so that a forgotten condition cannot mix forecasts into an analysis.
3. **Mean airborne speed.** Not a cleaning error, but a coverage rule the saved queries lack:
   a mean speed from fewer than 3 aircraft is noise (F-12). Add `MIN_AIRBORNE_SPEED` to the M5/M7 queries.

### Forward to modelling (M7)

- **Rows are too few and too clustered.** 79 usable snapshot-airport rows from 16 snapshots on 10 days. Split by
  day when validating, never by row; one airport has only 1 windy row (IST).
- **Control time of day and airport.** The gust peak and the snapshots both sit in the afternoon (F-05, F-06), so a
  model needs hour (or time band), airport and weekend as covariates instead of a windy/calm cut-off. Use the
  gust relative to the airport's own level.
- **Take the outcome carefully.** Use the airport-centred count of airborne aircraft; use mean speed only with at
  least 3 aircraft (F-12). Aircraft near an airport are counted for the nearest airport only.
- **Leakage and timing.** Use only observed weather hours (I-20), joined at the UTC hour of the snapshot; never
  a gust from after the snapshot. Do not use `n_ground` or counts derived from the same snapshot as features.
- **Seasonality and events.** Only mid-October to early November, with one storm day (27 Oct) that no snapshot
  caught; any model may be learning autumn, not wind. Snapshots at night, in the morning and on holidays
  are needed before M7 can answer anything about them.

## 6. Reproducibility check

Nothing above may have changed the database. This cell must pass for the notebook to count as run.

In [ ]:
fp_after = explore.db_fingerprint(con)
explore.assert_unchanged(fp_before, fp_after)
print("OK: all", len(fp_after), "table fingerprints are identical before and after the notebook")
con.close()